<a href="https://colab.research.google.com/github/tanstudent60/ai-movie-recommendation/blob/main/loan_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [32]:
# Added to resolve ImportError with gradio
!pip install --upgrade --force-reinstall huggingface_hub<0.20
import gradio as gr
import requests
import json
import time
from google.colab import userdata # Import userdata to access Colab secrets

# Function to call the Gemini API for loan risk prediction
def predict_loan_risk(
    applicant_age: int,
    applicant_income: float,
    loan_amount: float,
    loan_term: int,
    credit_score: int,
    employment_length: int,
    home_ownership: str,
    loan_purpose: str,
    existing_monthly_debts: float
) -> str:
    """
    Predicts loan risk using the Gemini API based on provided loan application details.

    Args:
        applicant_age (int): Age of the applicant in years.
        applicant_income (float): Annual income of the applicant.
        loan_amount (float): Requested loan amount.
        loan_term (int): Loan term in months.
        credit_score (int): Credit score of the applicant (e.g., FICO).
        employment_length (int): Employment length in years.
        home_ownership (str): Type of home ownership (Rent, Own, Mortgage, Other).
        loan_purpose (str): Purpose of the loan (Debt Consolidation, Education, Home Improvement, Venture, Other).
        existing_monthly_debts (float): Total existing monthly debt payments.

    Returns:
        str: A formatted string containing the predicted loan risk and justification.
    """
    # Retrieve the API key from Colab secrets using userdata
    # Your secret name is 'GOOGLE_API_KEY'
    api_key = userdata.get("GOOGLE_API_KEY")

    if not api_key:
        return "Error: Gemini API key not found. Please ensure your secret 'GOOGLE_API_KEY' is set up in Colab secrets and notebook access is enabled."

    # Construct the prompt for the Gemini model
    prompt = f"""
    Analyze the following loan application details and predict the loan risk as "Low Risk", "Medium Risk", or "High Risk".
    Provide a brief justification for your prediction.

    Loan Application Details:
    - Applicant Age: {applicant_age} years
    - Applicant Income: ${applicant_income} per year
    - Loan Amount: ${loan_amount}
    - Loan Term: {loan_term} months
    - Credit Score: {credit_score} (out of 850)
    - Employment Length: {employment_length} years
    - Home Ownership: {home_ownership}
    - Loan Purpose: {loan_purpose}
    - Existing Monthly Debts: ${existing_monthly_debts}

    Based on these details, what is the loan risk (Low Risk, Medium Risk, or High Risk) and why?
    Please format your response as follows:
    Risk: [Your Prediction]
    Justification: [Your Justification]
    """

    chat_history = [{"role": "user", "parts": [{"text": prompt}]}]

    payload = {
        "contents": chat_history,
        "generationConfig": {
            "responseMimeType": "text/plain",
        },
    }

    api_url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={api_key}"


    retries = 0
    max_retries = 5
    base_delay = 1  # 1 second

    while retries < max_retries:
        try:
            response = requests.post(
                api_url,
                headers={'Content-Type': 'application/json'},
                data=json.dumps(payload)
            )
            response.raise_for_status()  # Raise an HTTPError for bad responses (4xx or 5xx)

            result = response.json()

            if result.get("candidates") and len(result["candidates"]) > 0 and \
               result["candidates"][0].get("content") and \
               result["candidates"][0]["content"].get("parts") and \
               len(result["candidates"][0]["content"]["parts"]) > 0:
                text = result["candidates"][0]["content"]["parts"][0]["text"]

                # Parse the response to extract Risk and Justification
                risk_match = next((line for line in text.splitlines() if line.startswith("Risk:")), None)
                justification_match = next((line for line in text.splitlines() if line.startswith("Justification:")), None)

                prediction_text = "N/A"
                justification_text = "N/A"

                if risk_match:
                    prediction_text = risk_match.replace("Risk:", "").strip()
                if justification_match:
                    justification_text = justification_match.replace("Justification:", "").strip()

                return (
                    f"**Predicted Risk:** {prediction_text}\n\n"
                    f"**Justification:** {justification_text}"
                )
            else:
                return "Error: Unexpected API response structure." + str(result)

        except requests.exceptions.RequestException as e:
            print(f"Error calling Gemini API: {e}")
            retries += 1
            if retries < max_retries:
                delay = base_delay * (2 ** (retries - 1))
                print(f"Retrying in {delay} seconds...")
                time.sleep(delay)
            else:
                return f"Error: Failed to predict loan risk after multiple retries. {e}"
        except json.JSONDecodeError:
            return "Error: Could not decode JSON response from API."
        except Exception as e:
            return f"An unexpected error occurred: {e}"

    return "Error: Failed to predict loan risk after multiple retries. Please try again later."


# Define Gradio Interface
iface = gr.Interface(
    fn=predict_loan_risk, # Changed to directly use predict_loan_risk
    inputs=[
        gr.Number(label="Applicant Age (years)", minimum=18, value=30),
        gr.Number(label="Annual Applicant Income ($)", minimum=0, value=60000),
        gr.Number(label="Loan Amount ($)", minimum=1, value=15000),
        gr.Number(label="Loan Term (months)", minimum=1, value=36),
        gr.Number(label="Credit Score (300-850)", minimum=300, maximum=850, value=720),
        gr.Number(label="Employment Length (years)", minimum=0, value=5),
        gr.Dropdown(
            ["Rent", "Own", "Mortgage", "Other"],
            label="Home Ownership",
            value="Rent"
        ),
        gr.Dropdown(
            ["Debt Consolidation", "Education", "Home Improvement", "Venture", "Other"],
            label="Loan Purpose",
            value="Debt Consolidation"
        ),
        gr.Number(label="Existing Monthly Debts ($)", minimum=0, value=500),
    ],
    outputs=gr.Markdown(label="Loan Risk Prediction"),
    title="Gen AI Loan Risk Predictor",
    description="Enter the loan application details to get a risk assessment using Generative AI (Gemini).",
    css="""
    body { font-family: 'Inter', sans-serif; }
    .gradio-container { border-radius: 12px; box-shadow: 0 4px 6px rgba(0, 0, 0, 0.1); }
    h1 { color: #1f2937; font-weight: 800; }
    p { color: #4b5563; }
    button { background-color: #3b82f6 !important; color: white !important; border-radius: 0.375rem !important; }
    button:hover { background-color: #2563eb !important; }
    input[type=\"number\"], select { border-radius: 0.375rem; border: 1px solid #d1d5db; padding: 0.5rem 0.75rem; }
    input[type=\"number\"]:focus, select:focus { outline: none; border-color: #3b82f6; ring: 2px; ring-color: #93c5fd; }
    .gr-box { border-radius: 0.5rem; }
    .gr-button { border-radius: 0.5rem; }
    """
    # Removed flagging_mode="never" to resolve the AttributeError
)

# Launch the Gradio app
if __name__ == "__main__":
    iface.launch(share=True) # Set share=True to get a public link (useful for testing)

/bin/bash: line 1: 0.20: No such file or directory
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://dff52747d1c5bc4439.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [33]:
# Test API directly before launching UI
test_result = predict_loan_risk(30, 60000, 15000, 36, 720, 5, "Rent", "Education", 500)
print(test_result)

**Predicted Risk:** Low Risk

**Justification:** The applicant demonstrates several strong indicators of creditworthiness. They have a solid income ($60,000) and a remarkably low existing debt-to-income ratio. Their credit score (720) is excellent, reflecting a history of responsible financial behavior. Furthermore, their 5 years of employment length indicates stability, and the loan amount ($15,000) is relatively modest compared to their annual income. Even as a renter, the overall financial picture suggests a high likelihood of timely repayment.


In [31]:
from google.colab import userdata
import requests, json

api_key = userdata.get("GOOGLE_API_KEY")
print("Key starts with:", api_key[:8])  # just to confirm correct key loaded

url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={api_key}"
payload = {"contents": [{"role": "user", "parts": [{"text": "Say hello"}]}]}
response = requests.post(url, headers={"Content-Type": "application/json"}, data=json.dumps(payload))

print("Status:", response.status_code)
print("Response:", response.json())

Key starts with: AQ.Ab8RN
Status: 200
Response: {'candidates': [{'content': {'parts': [{'text': 'Hello!'}], 'role': 'model'}, 'finishReason': 'STOP', 'index': 0}], 'usageMetadata': {'promptTokenCount': 3, 'candidatesTokenCount': 2, 'totalTokenCount': 25, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 3}], 'thoughtsTokenCount': 20, 'serviceTier': 'standard'}, 'modelVersion': 'gemini-2.5-flash', 'responseId': 'L6ofasn-FNmGqtsPhsjekQU'}
